In [1]:
import SimplicialComplex as sc
import json
import sympy as sp
import numpy as np
from IPython.display import display
from itertools import combinations,permutations

def read_file(filename):
    with open(filename, 'rb') as f:
        data = f.readlines()
        data = [x.strip() for x in data]
    return data

def load_seeds(n,m):
    db_path = 'final_results/CSPLS_%d_%d' % (n, m)
    list_facets = [json.loads(facets_bytes) for facets_bytes in read_file(db_path)]
    return [sc.PureSimplicialComplex(facets_set) for facets_set in list_facets]

### Function for testing primitive collections

In [14]:
from sage.all import *

def enumerate_inequalities(K:sc.SimplicialComplex,char_map):
    m=K.m
    n=K.n
    K.compute_MNF_set()
    K.MNF_bin_to_MNF()
    facets_list = [np.array(sc.binary_to_face_0(facet_bin,m)) for facet_bin in K.facets_bin]
    N = len(facets_list)
    inequality_matrix = np.zeros((len(K.MNF_set),m))
    for l in range(len(K.MNF_set)):
        sigma = K.MNF_set[l]
        vec_char_sigma = np.zeros((m,1))
        vec_char_sigma[[i-1 for i in sigma],:] = 1
        x = np.dot(char_map,vec_char_sigma)
        for k in range(0,N): # Trouver dans quel cone il est
            y = np.dot(np.linalg.inv(char_map[:,facets_list[k]]),x)
            if np.all(y>=0):
                inequality_matrix[l,:] = vec_char_sigma.T.copy()
                for j in range(n):
                    if y[j][0]!=0:
                        inequality_matrix[l][facets_list[k][j]] = -y[j][0]
                break
    return inequality_matrix


def find_convex(K,char_map):
    (m,n) = (K.m,K.n)
    A = matrix(QQ, n, m, lambda i, j: 1 if i == j else 0)
    b = vector(QQ, [0] * n)

    # Create the polyhedron using the equality constraints
    P = Polyhedron(eqns=[(b[i],) + tuple(-A.row(i)) for i in range(n)])
    M=enumerate_inequalities(K,char_map)
    B = np.zeros((M.shape[0],M.shape[1]+1))
    B[:,1:m+1] = M
    list_of_lists_B = B.tolist()
    sage_set_B = Set(vector(ZZ, row) for row in list_of_lists_B)
    Q = Cone([r for r in (Polyhedron(ieqs=sage_set_B) & P).Vrepresentation()])
    # h=200*Q.Hilbert_basis()[0] + 111*Q.Hilbert_basis()[1]
    # h_np=np.array(h)
    for h in Q.Hilbert_basis():
        h_np=np.array(h)
        list_V = []
        facets_list = [np.array(sc.binary_to_face_0(facet_bin,m)) for facet_bin in K.facets_bin]
        for facet in facets_list:
            list_V.append(vector(ZZ,list(np.dot(np.linalg.inv(char_map[:,facet].T),h_np[facet]))))
        print(list_V)
        R = Polyhedron(vertices=list_V)
        print(NormalFan(R).rays())
        S = Polyhedron(ieqs=[(h[i],) + tuple(list(char_map[:,i].T)) for i in range(m)])
        print(NormalFan(S).rays())
        print(len(S.Vrepresentation()),len(K.facets))
        



n=8
m= n+4
K_7_58 = sc.PureSimplicialComplex(None,[[3,4,11],[3,4,12],[1,4,5,9],[1,4,5,12],[1,4,9,11],[2,5,8,12],[3,6,10,11],[3,8,10,12],[1,2,5,7,9],[1,6,7,9,11],[2,6,7,8,10],[6,7,9,10,11]],8)
char_map_1 = np.zeros((n,m))
char_map_1[:,:n]=np.eye(n)
char_map_1[:,n:] = np.array([[-1,0,0,-1],[-2,-1,2,-1],[1,0,-1,0],[1,1,-1,0],[-1,0,1,-1],[-1,-1,0,-1],[-2,-1,1,-1],[-1,-1,1,-1]])
find_convex(K_7_58,char_map_1)

[(0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, -1, 0, 0, 0, 0), (0, 0, -1, 0, 0, 0, 0, 0), (0, 0, 0, -2, 0, 0, -1, 0), (-1, 0, 0, -1, 0, 0, 0, 0), (-1, 0, -1, 0, 0, 0, 0, 0), (0, 0, -1, 0, 0, 0, 0, 0), (0, 0, -1, 0, 0, 0, 0, 0), (0, 0, 0, -1, 0, -1, 0, 0), (0, 0, -1, 0, 0, 0, 0, 0), (0, 0, -1, 0, 0, 0, 0, 0), (0, 0, 0, -1, 0, -1, 0, 0), (0, 0, -1, -1, 0, 0, -1, 0), (-1, 0, -1, 0, 0, 0, 0, 0), (0, 0, 0, -1, 0, -1, 0, 0), (0, 0, 0, -1, 0, -1, 0, 0), (0, 0, 0, -1, 0, -1, 0, 0), (-1, 0, -1, 0, 0, 0, 0, 0), (0, 0, 0, -1, -1, 0, 0, 0), (0, 0, -1, 0, -1, 0, 0, 0), (0, -1, 0, -2, 0, 0, 0, 0), (0

In [17]:
P = Polyhedron(vertices=[(0,0),(1,0),(0,1)])
print(NormalFan(P).rays())

N( 1,  0),
N( 0,  1),
N(-1, -1)
in 2-d lattice N
